# Paso 2 - Baseline v0 (percentiles robustos, sin etiquetas)

**KPCL0034 (comida, unificado) + KPCL0035 (agua)** - sin etiquetas de eventos,
el baseline tiene que ser auto-referenciado.

```
TODA LA SENIAL (post-dedup, ambos dispositivos)
        |
        v
Baseline v0: percentiles robustos sobre el 100%
   (mediana, MAD, P90/P95/P99 de |delta_peso| y velocidad)
        |
        v
   [siguiente iteracion, no en este notebook todavia:]
   excluir cola extrema (>P99) -> recalcular v1 -> comparar v0 vs v1
```

Este notebook implementa **solo Baseline v0** - percentiles robustos sobre el
100% de la senial, por `device_code`. La exclusion iterativa (v1) y la
normalizacion de duracion de estabilidad por cadencia quedan para el siguiente
paso, una vez que v0 este confirmado.

Carga desde `data/lecturas_limpias.csv`, el cache que escribe
`01_caracterizacion_fondo.ipynb` (Paso 1) al final de su celda de carga --
**correr ese notebook primero** si el cache no existe todavia.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CACHE_CSV = NOTEBOOK_DIR / "data" / "lecturas_limpias.csv"
GAP_CUTOFF_S = 300

# --- Carga desde cache (post-dedup, generado por 01_caracterizacion_fondo.ipynb) ---
# Si no existe, correr ese notebook primero -- el dedup de abril vive ahi, no aca.
df = pd.read_csv(CACHE_CSV)
df["device_id"] = df["device_id"].astype("category")
df["device_code"] = df["device_code"].astype("category")
df["ts"] = pd.to_datetime(df["ts"], format="ISO8601", utc=True)

df["delta_peso"] = df.groupby("device_id", observed=True)["peso"].diff()
df["delta_t"] = df.groupby("device_id", observed=True)["ts"].diff().dt.total_seconds()
df["abs_delta_peso"] = df["delta_peso"].abs()
df["delta_peso_valido"] = df["delta_peso"].where(df["delta_t"] <= GAP_CUTOFF_S)
df["velocidad_peso"] = df["delta_peso_valido"] / df["delta_t"]

print(f"Lecturas totales (post-dedup, desde cache): {len(df):,}")
print(df["device_code"].value_counts())


## Baseline v0

Percentiles robustos sobre el **100%** de la senial (sin excluir nada todavia)
- mediana y MAD como resumen central+dispersion (robustos a outliers, a
diferencia de media/STD), y P90/P95/P99 de `|delta_peso|` y `velocidad_peso`
como candidatos a umbral de "esto ya no es fondo".


In [ ]:
def baseline_v0(grupo: pd.DataFrame) -> dict:
    """Percentiles robustos sobre el 100% de un grupo (device_code) -- sin
    excluir nada. La exclusion iterativa (v1) es el siguiente paso, no este."""
    _abs_delta = grupo["abs_delta_peso"].dropna()
    _vel = grupo["velocidad_peso"].abs().dropna()
    _mediana_peso = grupo["peso"].median()
    return {
        "n": len(grupo),
        "mediana_peso_g": round(_mediana_peso, 2),
        "mad_peso_g": round((grupo["peso"] - _mediana_peso).abs().median(), 2),
        "abs_delta_mediana_g": round(_abs_delta.median(), 3),
        "abs_delta_mad_g": round((_abs_delta - _abs_delta.median()).abs().median(), 3),
        "abs_delta_p90_g": round(_abs_delta.quantile(0.90), 3),
        "abs_delta_p95_g": round(_abs_delta.quantile(0.95), 3),
        "abs_delta_p99_g": round(_abs_delta.quantile(0.99), 3),
        "velocidad_mediana_g_s": round(_vel.median(), 4),
        "velocidad_p90_g_s": round(_vel.quantile(0.90), 4),
        "velocidad_p95_g_s": round(_vel.quantile(0.95), 4),
        "velocidad_p99_g_s": round(_vel.quantile(0.99), 4),
    }

filas_v0 = []
for _code, _grupo in df.groupby("device_code", observed=True):
    _fila = baseline_v0(_grupo)
    _fila["device_code"] = _code
    filas_v0.append(_fila)

baseline_v0_df = pd.DataFrame(filas_v0).set_index("device_code")
print("--- Baseline v0 (100% de la senial, por device_code) ---")
baseline_v0_df


## Lectura de Baseline v0

`abs_delta_p99_g` y `velocidad_p99_g_s` son los primeros candidatos a umbral
de "esto ya no es fondo, es actividad" -- por ejemplo, el umbral actual del
detector de candidatos (`01_genera_candidatos.py`, `umbral_delta_g=5.0`) se
puede comparar directo contra el P99 de acá: si el P99 real de KPCL0034 queda
muy por debajo o muy por encima de 5.0g, es evidencia de que el umbral fijo
actual esta mal calibrado contra el fondo real medido.

**Ojo:** v0 se calculo sobre el 100% de la senial, sin excluir nada -- ese
P99 es "fondo + la cola de eventos reales que ya esten contaminando el
percentil 99", probablemente un **limite superior** del ruido de fondo real,
no el fondo puro todavia. Por eso importa v1 (siguiente celda), no es un
refinamiento cosmetico.

**Pendiente, no en este notebook:** normalizacion de duracion de estabilidad
por cadencia (`duracion_s / cadencia_mediana_del_device`) antes de
incorporarla al baseline.


In [ ]:
# === Baseline v1: excluir >P99 de v0 por lectura individual, recalcular ======
# Criterio de exclusion: por lectura individual (no por corrida/ventana) --
# es el mas simple y el menos propenso a excluir de mas. Una sola iteracion
# (v0 -> v1): si v1 cambia poco respecto a v0, el fondo ya es estable y no
# hace falta iterar de nuevo; si cambia bastante, recien ahi valdria la pena
# una v2.
filas_v1 = []
for _code, _grupo in df.groupby("device_code", observed=True):
    _p99_v0 = _grupo["abs_delta_peso"].quantile(0.99)
    _grupo_v1 = _grupo[_grupo["abs_delta_peso"] <= _p99_v0]

    _mediana_peso_v0 = _grupo["peso"].median()
    _mad_v0 = (_grupo["peso"] - _mediana_peso_v0).abs().median()
    _mediana_peso_v1 = _grupo_v1["peso"].median()
    _mad_v1 = (_grupo_v1["peso"] - _mediana_peso_v1).abs().median()
    _p99_v1 = _grupo_v1["abs_delta_peso"].quantile(0.99)

    print(f"--- {_code} ---")
    print(f"n v0: {len(_grupo):,}  ->  n v1: {len(_grupo_v1):,} "
          f"({(1 - len(_grupo_v1) / len(_grupo)) * 100:.2f}% excluido)")
    print(f"MAD peso     v0: {_mad_v0:.2f}g  ->  v1: {_mad_v1:.2f}g")
    print(f"|delta_peso| P99  v0: {_p99_v0:.3f}g  ->  v1: {_p99_v1:.3f}g")
    print()

    filas_v1.append({
        "device_code": _code,
        "pct_excluido": round((1 - len(_grupo_v1) / len(_grupo)) * 100, 2),
        "mad_peso_v0": round(_mad_v0, 2), "mad_peso_v1": round(_mad_v1, 2),
        "abs_delta_p99_v0": round(_p99_v0, 3), "abs_delta_p99_v1": round(_p99_v1, 3),
    })

baseline_v1_df = pd.DataFrame(filas_v1).set_index("device_code")
baseline_v1_df


## Lectura de Baseline v1 - hallazgo negativo, pero util

**Resultado real (2026-08-29):** `abs_delta_p99` colapsa de 3.0g/2.0g (v0) a
**0.000g exacto** (v1) en los dos dispositivos, excluyendo apenas 1.08%/0.84%
de las filas. No es un refinamiento gradual - es un colapso total. Vale la
pena dejar el razonamiento completo, no solo el resultado, para que esto no
se re-intente en un ciclo futuro sin saber que ya se agoto por esta via.

**Por que pasa (mecanico, no un bug):** ya sabiamos de Paso 1 que
`delta_peso == 0` es ~98% de la senial. El 1% que excluye v0 (top percentil
de `|delta_peso|`) es, en la practica, casi TODA la fraccion de lecturas que
alguna vez se movio - no una porcion de esa fraccion. Al sacarla, lo que
queda es casi puro "no se movio nada", y su propio P99 da 0 por definicion.
El colapso a 0 es matematicamente inevitable dado ese ~98% de ceros - **no es
evidencia de que el sensor no tenga ruido real**, es evidencia de que la
magnitud de una sola lectura no es la variable que separa ruido de evento a
esta resolucion (1g entero, ver Paso 1 bloque 3).

**El metodo en si funciono bien** - la prueba es que `mad_peso` **no se movio
ni un gramo** (22.00->22.00g KPCL0034, 63.00->63.00g KPCL0035) al excluir ese
1%. Eso confirma que lo excluido era cola pura, sin arrastrar la dispersion
general - el calculo llego a un **limite estructural de la variable elegida**,
no a un error de calculo ni un bug de indexacion.

**Conclusion metodologica (el cierre mas valioso de este bloque):** con esta
resolucion de sensor, no hay una capa intermedia de "ruido ancho pero todavia
no evento" para que una segunda iteracion (v1->v2) siga refinando - v1 ya es
degenerado (todo cero), iterar de nuevo converge trivialmente a 0 otra vez.
**Percentiles de magnitud a nivel de UNA lectura individual quedan descartados
como via metodologica para separar ruido de evento en este dataset** - un
evento real no es "una lectura con delta_peso grande", es "una secuencia de
lecturas con un patron temporal distinto al fondo".

Esto le da **respaldo empirico retroactivo** a una decision de diseno que ya
existe en el motor real: `shape_features_v2.py` (`Ciclo_Alpha_v2/fase_0_ruido/`)
nunca clasifico por umbral de magnitud de una lectura - fue directo a 102
features de **forma y duracion del segmento completo** (F00-F14, ver
[[11_ModelosIA/MODEL_EvidenceEngine]]). Hoy confirmamos con datos por que ese
camino era necesario, no solo preferible.

**Siguiente paso (continuacion directa, no una rama alternativa):**
normalizacion de duracion de estabilidad por cadencia - es la variable
posicionada para empezar a capturar la forma/duracion del segmento en vez de
la magnitud de un punto aislado, siguiendo la misma logica que ya usa el motor
real.


## Normalizacion de duracion de estabilidad por cadencia

Unidad: **numero de intervalos de muestreo esperados**
(`duracion_real_s / cadencia_mediana_de_ESE_device_id`) - no segundos crudos
ni conteo de filas. Una corrida de 90s a cadencia 15s (6 intervalos) y una de
180s a cadencia 30s (tambien 6 intervalos) quedan como el mismo numero,
comparable entre abril y mayo+ sin depender todavia de decidir fondo separado
o unificado para esta metrica.

`corrida_id` se agrupa por `device_id` (UUID), no `device_code`: cada corrida
ya queda confinada a un solo UUID de todos modos (`delta_peso` es `NaN` en la
primera fila de cada UUID nuevo, que corta la corrida ahi solo) - agrupar por
`device_id` deja el resultado listo para dividir por SU propia cadencia.


In [ ]:
is_gap = df["delta_t"] > GAP_CUTOFF_S
paso_estable = (df["delta_peso"] == 0) & (~is_gap.fillna(False))
corrida_id = (~paso_estable).groupby(df["device_id"], observed=True).cumsum()
duraciones = df["delta_t"][paso_estable].groupby(
    [df.loc[paso_estable, "device_id"], corrida_id[paso_estable]], observed=True,
).sum()

cadencia_por_device = df.groupby("device_id", observed=True)["delta_t"].median()
duraciones_normalizadas = duraciones / duraciones.index.get_level_values(0).map(cadencia_por_device).astype(float).to_numpy()

for _uuid, _periodo in [
    ("9510a455-b0e9-4932-8be1-03976d31228a", "KPCL0034 - abril"),
    ("3a460074-e7c3-41bf-ae5a-a011445f927a", "KPCL0034 - mayo-agosto"),
    ("0dc601c0-1533-40c5-b606-6d89eb2d4042", "KPCL0035"),
]:
    _en_indice = _uuid in duraciones.index.get_level_values(0)
    _dur_s = duraciones.loc[_uuid] if _en_indice else pd.Series(dtype=float)
    _dur_n = duraciones_normalizadas.loc[_uuid] if _en_indice else pd.Series(dtype=float)
    print(f"--- {_periodo} ---")
    print(f"cadencia mediana: {cadencia_por_device.get(_uuid, float('nan')):.2f}s")
    if len(_dur_s) > 0:
        print(f"duracion (s)          P50/P90/P95: {_dur_s.quantile([.5, .9, .95]).round(1).tolist()}")
        print(f"duracion (intervalos) P50/P90/P95: {_dur_n.quantile([.5, .9, .95]).round(2).tolist()}")
    else:
        print("sin corridas encontradas")
    print()


## Lectura de la duracion normalizada - la hipotesis de Paso 1 no se sostuvo

**Resultado real (2026-08-29):**

| | cadencia mediana | duracion (intervalos) P90 | P95 |
|---|---|---|---|
| KPCL0034 - abril | 30.00s | **392.1** | 531.0 |
| KPCL0034 - mayo-agosto | 30.00s | **192.2** | 336.6 |

**La cadencia mediana post-dedup ya es identica (30.00s ambos)** - por lo tanto
normalizar dividiendo por la cadencia mediana no cambia nada respecto a
trabajar en segundos crudos, y la diferencia de ~2x en P90 **sigue exactamente
igual** (392 vs 192 intervalos, la misma razon que 11.761s vs 5.767s en Paso 1).

**Esto contradice la hipotesis que dejamos escrita en el cierre de Paso 1**
("artefacto residual de cadencia que el dedup no corrige"). No era eso -
la cadencia ya estaba corregida, y aun asi la duracion de estabilidad difiere
genuinamente entre periodos. Se deja constancia del error de hipotesis en vez
de forzar la conclusion original: **hay una diferencia de comportamiento real
entre abril y mayo-agosto** en cuanto dura el bowl sin tocarse (posiblemente
rutina de alimentacion distinta entre esos meses, o algo todavia no explicado)
- no cadencia, no tara/hardware (ya descartado antes).

**Decision para el baseline de estabilidad:** no se puede unificar KPCL0034
como un solo periodo para esta metrica especifica. El baseline de "cuanto dura
normalmente estar quieto" queda **separado por periodo** hasta investigar la
causa real de esta diferencia (fuera de alcance de este Paso 2).


## Resumen final (tabla y gráfico)

Consolida el colapso v0→v1 (por `device_code`) y la duración normalizada
(por período) en una sola tabla y un solo gráfico — reutiliza
`baseline_v0_df`, `baseline_v1_df`, `duraciones` y `duraciones_normalizadas`
ya calculados arriba.


In [ ]:
import matplotlib.pyplot as plt

# --- Tabla resumen: v0/v1 por device_code + duracion normalizada por periodo ---
_periodos_resumen = [
    ("KPCL0034 - abril", "9510a455-b0e9-4932-8be1-03976d31228a", "KPCL0034"),
    ("KPCL0034 - mayo-agosto", "3a460074-e7c3-41bf-ae5a-a011445f927a", "KPCL0034"),
    ("KPCL0035", "0dc601c0-1533-40c5-b606-6d89eb2d4042", "KPCL0035"),
]

filas_resumen = []
for _nombre, _uuid, _device_code in _periodos_resumen:
    _dur_n = duraciones_normalizadas.loc[_uuid] if _uuid in duraciones.index.get_level_values(0) else pd.Series(dtype=float)
    filas_resumen.append({
        "periodo": _nombre,
        "mad_peso_v0_g": baseline_v0_df.loc[_device_code, "mad_peso_g"],
        "abs_delta_p99_v0_g": baseline_v1_df.loc[_device_code, "abs_delta_p99_v0"],
        "abs_delta_p99_v1_g": baseline_v1_df.loc[_device_code, "abs_delta_p99_v1"],
        "duracion_p90_intervalos": round(_dur_n.quantile(0.90), 1) if len(_dur_n) else float("nan"),
        "duracion_p95_intervalos": round(_dur_n.quantile(0.95), 1) if len(_dur_n) else float("nan"),
    })

resumen_final_df = pd.DataFrame(filas_resumen).set_index("periodo")
print("--- Resumen final Paso 2 ---")
print(resumen_final_df)

# --- Grafico resumen: 3 paneles ------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
_device_codes = baseline_v1_df.index.tolist()
_x = np.arange(len(_device_codes))

# Panel 1: colapso de abs_delta_p99 (v0 -> v1)
axes[0].bar(_x - 0.2, baseline_v1_df["abs_delta_p99_v0"], width=0.4, label="v0", color="#2980b9")
axes[0].bar(_x + 0.2, baseline_v1_df["abs_delta_p99_v1"], width=0.4, label="v1", color="#c0392b")
axes[0].set_xticks(_x)
axes[0].set_xticklabels(_device_codes)
axes[0].set_ylabel("|delta_peso| P99 (g)")
axes[0].set_title("Colapso v0 -> v1 (percentil de magnitud)")
axes[0].legend()

# Panel 2: MAD de peso, sin cambios (evidencia de que el metodo funciono)
axes[1].bar(_x - 0.2, baseline_v1_df["mad_peso_v0"], width=0.4, label="v0", color="#2980b9")
axes[1].bar(_x + 0.2, baseline_v1_df["mad_peso_v1"], width=0.4, label="v1", color="#c0392b")
axes[1].set_xticks(_x)
axes[1].set_xticklabels(_device_codes)
axes[1].set_ylabel("MAD de peso (g)")
axes[1].set_title("MAD sin cambios v0 -> v1")
axes[1].legend()

# Panel 3: duracion normalizada P90/P95 por periodo
_x3 = np.arange(len(resumen_final_df))
axes[2].bar(_x3 - 0.2, resumen_final_df["duracion_p90_intervalos"], width=0.4, label="P90", color="#27ae60")
axes[2].bar(_x3 + 0.2, resumen_final_df["duracion_p95_intervalos"], width=0.4, label="P95", color="#8e44ad")
axes[2].set_xticks(_x3)
axes[2].set_xticklabels(resumen_final_df.index, rotation=20, ha="right")
axes[2].set_ylabel("Duracion (intervalos de muestreo)")
axes[2].set_title("Duracion normalizada por periodo")
axes[2].legend()

fig.suptitle("Paso 2 - Resumen: colapso v0/v1 y duracion normalizada")
fig.tight_layout()
plt.show()
